#### 1.0 Libraries and directories

In [1]:
import ee 
import geemap
import geopandas as gpd
import pandas as pd
import datetime as dt
import pprint as pp
from shapely.geometry import shape

ee.Authenticate()
ee.Initialize(project='ee-green-by-another-name')

roi_name = 'YKF_sub1'
level = 'sr' # 1) 'sr': surface reflectance 2) 'toa' for top of atmosphere
resample_res = 30 
resample_method = 'bilinear'
n_dates = 5 # Number of observation dates
s2_cloud_threshold = 20 # Cloud probability threshold for Sentinel-2
band_dict = {'Sentinel-2': ['B2', #Blue
                            'B3', #Green
                            'B4', #Red
                            'B8'], #NIR
             'LandSat8': ['SR_B2', #Blue
                          'SR_B3', #Green
                          'SR_B4', #Red
                          'SR_B5'] #NIR
}

image_footprints_path = f'./data/overlap_dates_for_roi/{roi_name}_overlap_dates.shp'
best_image_dates = gpd.read_file(image_footprints_path) 
est_utm = f'EPSG:{best_image_dates.estimate_utm_crs().to_epsg()}' # Have to convert pyproj object into literal string for ee


rois UTM CRS is EPSG:32606


#### 2.0 Select target dates

In [2]:
best_image_dates['date_plus_1d'] = pd.to_datetime(best_image_dates['date']) + pd.Timedelta(days=1)
best_image_dates = best_image_dates[1:n_dates]

print(best_image_dates[['date', 'per_cover']])

         date  per_cover
1  2021-07-01       81.0
2  2024-07-25       81.0
3  2017-09-08       80.0
4  2022-06-09       73.0


In [ ]:
######################################
# Misc. helper functions
######################################
def convert_gpd_geom_to_ee(geom):
        """
        Takes a geopandas geom object and coverts it to an Earth Engine polygon
        """
        coords = list(geom.exterior.coords)
        coords_list = [[x, y] for x, y in coords]
        return ee.Geometry.Polygon(coords_list, proj='EPSG:4326')

def add_1d_to_date(date: str):
        date_plus_1d = pd.to_datetime(date) + pd.Timedelta(days=1)
        date_plus_1d.strftime('%Y-%m-%d')
        return date_plus_1d

#########################################
# Part I: Functions to find the Sentinel-2 and Landsat8 image collections
#########################################

def find_s2_col(footprint: gpd.GeoSeries, level: str):

    date = footprint['date']
    date_plus_1d = add_1d_to_date(date)
    polygon = convert_gpd_geom_to_ee(footprint['geometry'])
    
    
    if level == 'sr':
        s2_string = 'COPERNICUS/S2_SR'
    elif level == 'toa':
        print('figure out TOA params lazy dummy')
    else:
        print(f'ERROR: level arg should be "sr" or "toa" not {level}')
    
    s2_col = (
        ee.ImageCollection(s2_string)
        .filterDate(date, date_plus_1d)
        .filterBounds(polygon)
    )

    if s2_col.size().getInfo() == 0:
        print(f'ERROR: No S2 images found for {date} with {level} processing level')
        return None
    
    return s2_col, polygon

def find_ls8_col(footprint: gpd.GeoSeries):
     return None

#########################################
# Part II: Functions to select bands and rescale numerical values
#########################################

def fetch_rescale_s2_imgs(s2_col: ee.ImageCollection,
                          polygon: ee.Geometry,
                          bands: list):
    """
    Generates a single image mosiac with desired bands
    Rescales the bands to match (0-1) surface reflectance range
    TODO: Is rescaling different for TOA??
    """
        
    s2_img = (s2_col.select(bands)
              .mosaic()
              .clip(polygon))
    
    def rescale_s2(img):
        rescaled_bands = img.divide(10_000)
        return rescaled_bands
    
    s2_img = rescale_s2(s2_img)

    return s2_img 

def fetch_rescale_ls8_imgs(ls8_col: ee.ImageCollection):
    """"""
    return None

#########################################
# Part III: Functions to produce individual and common cloud masks
#########################################
  
def make_s2_cloud_mask(footprint: gpd.GeoSeries, s2_col: ee.ImageCollection, s2_cloud_threshold: int):
    """
    Produces a binary cloud mask from the Copernicus Cloud Probability and Sentinel-2 SCL (Scene Classification Layer)
    """
    date = footprint['date']
    date_plus_1d = add_1d_to_date(date)
    polygon = convert_gpd_geom_to_ee(footprint['geometry'])

    s2_cloud_prob_string = 'COPERNICUS/S2_CLOUD_PROBABILITY'
    s2_clouds = (ee.ImageCollection(s2_cloud_prob_string)
                 .filterBounds(polygon)
                 .filterDate(date, date_plus_1d)
                 .mosaic()
                 .clip(polygon))
    
    # Select the SCL band from the Sentinel-2 image collection
    s2_scl = (s2_col.select('SCL')
              .mosaic()
              .clip(polygon))
    
    clouds_binary = s2_clouds.select('probability').gt(s2_cloud_threshold).rename('cl_binary')
    s2_shaddow_mask = s2_scl.eq(3)
    s2_cirrus_mask = s2_scl.eq(10) 
    s2_full_mask = clouds_binary.Or(s2_shaddow_mask).Or(s2_cirrus_mask)

    return s2_full_mask

def reduce_mask_resolution(mask: ee.Image, resample_res: int, est_utm: str):
    """Reduces the resolution of cloud masks"""

    print(est_utm)
    original_crs = mask.projection()
    mask_reproj = mask.reproject(
        crs=ee.Projection(est_utm),
        scale=resample_res
    )
    mask_repoj_reduced = mask_reproj.reduceResolution(
        reducer=ee.Reducer.mean()
    ).reproject(
        crs=original_crs,
        scale=resample_res
    )
    return mask_repoj_reduced

#########################################
# Part IV: Functions to produce individual and common cloud masks
#########################################

def resample_img_then_mask(img: ee.Image, 
                          common_mask: ee.Image, 
                          est_utm: str, 
                          resample_res: int,
                          resample_method: str):
    """
    Resample to match the common cloud mask, then mask the image
    """
    print(est_utm)
    reproj = img.reproject(
         crs=ee.Projection(est_utm),
         scale=resample_res
    )
    resamp = reproj.resample(resample_method).reproject(
         crs=img.projection(),
         scale=resample_res
    )
    
    masked = resamp.updateMask(common_mask.neq(1))
    
    return masked

#########################################
# Part V: Main processsing & export functions
#########################################

def s2_processor(footprint: gpd.GeoSeries, 
                level: str, 
                bands: list, 
                resample_res: int,
                resample_method: str,
                est_utm: str, 
                s2_cloud_threshold: int):
    """
    Main function to process and export the Sentinel-2 images
    """
    s2_col, footprint_geom_ee = find_s2_col(footprint, level)
    if s2_col is None:
        return None
        
    s2_img = fetch_rescale_s2_imgs(s2_col, footprint_geom_ee, bands)
    s2_cloud_mask = make_s2_cloud_mask(footprint, s2_col, s2_cloud_threshold)
    s2_cloud_mask = reduce_mask_resolution(s2_cloud_mask, resample_res, est_utm)
    masked_s2 = resample_img_then_mask(s2_img, s2_cloud_mask, est_utm, resample_res, resample_method)
    
    s2_export = ee.batch.Export.image.toDrive(
         image=masked_s2,
         description=f'Sentinel2-{footprint['date']}_{roi_name}',
         fileNamePrefix=f'Sentinel2-{footprint['date']}_{roi_name}',
         folder='scrap',
         scale=30,
         region=footprint_geom_ee,
         crs='EPSG:4326',
         fileFormat='GeoTIFF',
         maxPixels=1e13
    )

    s2_export.start()
    print('Exporting Sentinel-2')
    

def ls_processor(resample_res: int):
    """
    Main function to process and export LandSat-8 images
    """
    return None

def main_processor(some_bullshit):
    return None
    

In [ ]:
for idx, row in best_image_dates.iterrows():
    print(row['date'])
    s2_processor(row, 
                level=level, 
                bands=band_dict['Sentinel-2'], 
                resample_res=resample_res,
                resample_method=resample_method,
                est_utm=est_utm,
                s2_cloud_threshold=s2_cloud_threshold
                )